# 🔍 VecGlypher QC — Анализ и сборка шрифта
Используйте этот ноутбук для проверки качества SVG файлов и сборки шрифта.
Работает как в Google Colab так и в JupyterLab на RunPod.

---
## Ячейка 1 — Установка зависимостей
Запустите один раз.

In [ ]:
# Консоль/терминал:
# pip install fonttools ufo2ft booleanoperations ufoLib2 matplotlib

import subprocess
result = subprocess.run("pip install fonttools ufo2ft booleanoperations ufoLib2 matplotlib 2>&1 | tail -5",
                        shell=True, capture_output=True, text=True)
print(result.stdout)


---
## Ячейка 2 — Настройки
Укажите путь к папке с SVG файлами.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, HTML

# Определяем среду — Colab или JupyterLab
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Colab — Drive подключён")
except ImportError:
    print("✅ JupyterLab — работаем локально")

# ✏️ МЕНЯЙТЕ ЭТИ ПАРАМЕТРЫ
if IN_COLAB:
    SVG_DIR = "/content/drive/MyDrive/output"  # путь в Google Drive
else:
    SVG_DIR = "/workspace/VecGlypher/outputs"  # путь на RunPod

print(f"📁 Папка SVG: {SVG_DIR}")
print(f"   Существует: {os.path.isdir(SVG_DIR)}")


---
## Ячейка 3 — Сканирование SVG файлов

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

def find_svg_files(base_dir):
    base = Path(base_dir)
    results = {}
    root_svgs = sorted(base.glob("*.svg"))
    if root_svgs:
        results["(корень)"] = root_svgs
    for folder in sorted(base.iterdir()):
        if folder.is_dir():
            svgs = sorted(folder.glob("*.svg"))
            if svgs:
                results[folder.name] = svgs
    return results

def render_file_list(svg_map):
    total = sum(len(v) for v in svg_map.values())
    html = """
    <div style='background:#1e1e1e; padding:16px; border-radius:10px; font-family:monospace;'>
    """
    html += f"<h3 style='color:#e0e0e0; margin-top:0'>✅ Найдено {total} SVG файлов в {len(svg_map)} папках</h3>"
    for folder, files in svg_map.items():
        upper = [f for f in files if f.stem.rstrip("_").isupper() and len(f.stem.rstrip("_")) == 1]
        lower = [f for f in files if f.stem.endswith("_") or (f.stem.islower() and len(f.stem) == 1)]
        other = [f for f in files if f not in upper and f not in lower]
        html += f"<h4 style='color:#7ec8e3; margin-bottom:8px'>📁 {folder} — {len(files)} файлов</h4>"
        if upper:
            names = " ".join(f"<span style='display:inline-block;min-width:28px;text-align:center;background:#2a3f5f;color:#a8d8f0;border-radius:4px;padding:2px 6px;margin:2px;font-weight:bold'>{f.stem}</span>" for f in upper)
            html += f"<div style='margin-bottom:8px'><small style='color:#888'>🔡 Заглавные ({len(upper)}):</small><br>{names}</div>"
        if lower:
            names = " ".join(f"<span style='display:inline-block;min-width:28px;text-align:center;background:#3f3a20;color:#f0d080;border-radius:4px;padding:2px 6px;margin:2px'>{f.stem}</span>" for f in lower)
            html += f"<div style='margin-bottom:8px'><small style='color:#888'>🔤 Строчные ({len(lower)}):</small><br>{names}</div>"
        if other:
            names = " ".join(f"<span style='display:inline-block;min-width:28px;text-align:center;background:#2e2e2e;color:#cccccc;border-radius:4px;padding:2px 6px;margin:2px'>{f.stem}</span>" for f in other)
            html += f"<div style='margin-bottom:8px'><small style='color:#888'>📄 Другие ({len(other)}):</small><br>{names}</div>"
        html += "<hr style='border-color:#333; margin:12px 0'>"
    html += "</div>"
    return html

svg_map = find_svg_files(SVG_DIR)
if not svg_map:
    print("❌ SVG файлы не найдены!")
else:
    display(HTML(render_file_list(svg_map)))


---
## Ячейка 4 — Анализ качества + Просмотр + Удаление

In [ ]:
import xml.etree.ElementTree as ET
import re
import shutil
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def analyze_svg(filepath):
    problems = []
    warnings = []
    try:
        content = Path(filepath).read_text()
    except Exception as e:
        return [f"❌ Не удалось прочитать файл: {e}"], []
    if '"/>' not in content and '" />' not in content:
        problems.append("❌ Path обрван — достигнут лимит токенов (увеличьте MAX_TOKENS)")
        return problems, warnings
    try:
        root = ET.fromstring(content)
    except ET.ParseError as e:
        problems.append(f"❌ Невалидный XML: {e}")
        return problems, warnings
    ns = {"svg": "http://www.w3.org/2000/svg"}
    paths = root.findall(".//svg:path", ns) + root.findall(".//path")
    if not paths:
        problems.append("❌ Нет path элементов — пустой глиф")
        return problems, warnings
    for i, path in enumerate(paths):
        d = path.get("d", "").strip()
        if not d:
            problems.append(f"❌ Path #{i+1}: пустой атрибут d")
            continue
        commands = re.findall(r"[MmLlHhVvCcSsQqTtAaZz]", d)
        if len(commands) < 3:
            problems.append(f"❌ Path #{i+1}: слишком мало команд ({len(commands)})")
        numbers = [float(x) for x in re.findall(r"-?\d+\.?\d*", d)]
        if numbers:
            min_val, max_val = min(numbers), max(numbers)
            if max_val > 1500 or min_val < -500:
                warnings.append(f"⚠️ Path #{i+1}: координаты далеко за viewBox ({min_val:.0f}…{max_val:.0f})")
    return problems, warnings

# Запускаем анализ
all_results = {}
for folder, files in svg_map.items():
    for f in files:
        problems, warnings = analyze_svg(f)
        all_results[f] = {"problems": problems, "warnings": warnings, "ok": len(problems) == 0}

total = len(all_results)
good  = sum(1 for r in all_results.values() if r["ok"] and not r["warnings"])
warn  = sum(1 for r in all_results.values() if r["ok"] and r["warnings"])
bad   = sum(1 for r in all_results.values() if not r["ok"])
bad_files = [f for f, r in all_results.items() if not r["ok"]]
bad_chars = [f.stem.rstrip("_") for f in bad_files]

def svg_preview(filepath, size=90):
    try:
        content = Path(filepath).read_text()
        preview = re.sub(r'width="[^"]*"', f'width="{size}"', content)
        preview = re.sub(r'height="[^"]*"', f'height="{size}"', preview)
        return preview
    except:
        return f"<div style='width:{size}px;height:{size}px;background:#333;color:#f06060;display:flex;align-items:center;justify-content:center'>ERR</div>"

def make_analysis_html():
    html = f"""
    <div style='background:#1e1e1e; padding:16px; border-radius:10px; font-family:monospace;'>
    <h3 style='color:#e0e0e0; margin-top:0'>🔍 Анализ качества SVG</h3>
    <div style='margin-bottom:16px'>
        <span style='background:#1a3a1a;color:#6fcf6f;padding:4px 12px;border-radius:4px;margin-right:8px'>✅ Хорошие: {good}</span>
        <span style='background:#3a3a1a;color:#f0c060;padding:4px 12px;border-radius:4px;margin-right:8px'>⚠️ Предупреждения: {warn}</span>
        <span style='background:#3a1a1a;color:#f06060;padding:4px 12px;border-radius:4px'>❌ Проблемные: {bad}</span>
    </div>
    """
    for folder, files in svg_map.items():
        folder_bad  = [f for f in files if not all_results[f]["ok"]]
        folder_warn = [f for f in files if all_results[f]["ok"] and all_results[f]["warnings"]]
        folder_good = [f for f in files if all_results[f]["ok"] and not all_results[f]["warnings"]]
        html += f"<h4 style='color:#7ec8e3'>📁 {folder}</h4>"
        if folder_good:
            names = " ".join(f"<span style='background:#1a3a1a;color:#6fcf6f;border-radius:4px;padding:2px 6px;margin:2px;display:inline-block'>{f.stem}</span>" for f in folder_good)
            html += f"<div style='margin-bottom:8px'><small style='color:#888'>✅ OK ({len(folder_good)}):</small><br>{names}</div>"
        if folder_warn:
            names = " ".join(f"<span style='background:#3a3a1a;color:#f0c060;border-radius:4px;padding:2px 6px;margin:2px;display:inline-block'>{f.stem}</span>" for f in folder_warn)
            html += f"<div style='margin-bottom:8px'><small style='color:#888'>⚠️ ({len(folder_warn)}):</small><br>{names}</div>"
            for f in folder_warn:
                for w in all_results[f]["warnings"]:
                    html += f"<div style='color:#f0c060;font-size:0.85em;margin-left:16px'>{f.stem}: {w}</div>"
        if folder_bad:
            names = " ".join(f"<span style='background:#3a1a1a;color:#f06060;border-radius:4px;padding:2px 6px;margin:2px;display:inline-block'>{f.stem}</span>" for f in folder_bad)
            html += f"<div style='margin-bottom:8px'><small style='color:#888'>❌ ({len(folder_bad)}):</small><br>{names}</div>"
            for f in folder_bad:
                for p in all_results[f]["problems"]:
                    html += f"<div style='color:#f06060;font-size:0.85em;margin-left:16px'>{f.stem}: {p}</div>"
        html += "<hr style='border-color:#333; margin:12px 0'>"
    if bad_chars:
        html += f"<div style='color:#aaa;font-size:0.9em'>💡 Для перегенерации: <code style='color:#f0c060'>{chr(44).join(bad_chars)}</code></div>"
    html += "</div>"
    return html

def make_all_glyphs_html():
    all_files = [f for files in svg_map.values() for f in files]
    upper = [f for f in all_files if f.stem.rstrip("_").isupper() and len(f.stem.rstrip("_")) == 1]
    lower = [f for f in all_files if f.stem.endswith("_") or (f.stem.islower() and len(f.stem) == 1)]

    def glyph_grid(files):
        html = "<div style='display:flex;flex-wrap:wrap;gap:8px;padding:8px'>"
        for f in files:
            is_bad = not all_results[f]["ok"]
            border = "2px solid #f06060" if is_bad else "1px solid #333"
            bg = "#2a1a1a" if is_bad else "#2a2a2a"
            html += f"""<div style='text-align:center;background:{bg};border:{border};border-radius:6px;padding:4px'>
                {svg_preview(f, 80)}
                <div style='color:#{"f06060" if is_bad else "aaa"};font-size:0.8em'>{f.stem}</div>
            </div>"""
        html += "</div>"
        return html

    html = "<div style='background:#1e1e1e;padding:16px;border-radius:10px'>"
    html += f"<h3 style='color:#e0e0e0;margin-top:0'>👁️ Все глифы ({len(all_files)} шт.)</h3>"
    if upper:
        html += "<h4 style='color:#a8d8f0'>🔡 Заглавные</h4>"
        html += glyph_grid(upper)
    if lower:
        html += "<h4 style='color:#f0d080'>🔤 Строчные</h4>"
        html += glyph_grid(lower)
    html += "<small style='color:#555'>Красная рамка = проблемный файл</small></div>"
    return html

# Вкладка удаления
all_files_list = [f for files in svg_map.values() for f in files]
file_options = [(f"{f.stem} {'❌' if not all_results[f]['ok'] else '✅'}", str(f))
                for f in sorted(all_files_list, key=lambda x: x.stem)]

delete_select = widgets.SelectMultiple(options=file_options, description="Выбрать:",
    layout=widgets.Layout(width="350px", height="300px"),
    style={"description_width": "initial"})
select_bad_btn = widgets.Button(description="☑️ Выбрать все битые", button_style="warning",
                                layout=widgets.Layout(width="200px"))
delete_btn = widgets.Button(description="🗑️ Удалить выбранные", button_style="danger",
                            layout=widgets.Layout(width="200px"))
regen_btn = widgets.Button(description="🔄 Перегенерировать (скоро)", button_style="",
                           disabled=True, layout=widgets.Layout(width="220px"))
delete_out = widgets.Output()

def on_select_bad(b):
    bad_paths = [str(f) for f in bad_files]
    delete_select.value = [p for _, p in file_options if p in bad_paths]

def on_delete(b):
    with delete_out:
        delete_out.clear_output()
        selected = delete_select.value
        if not selected:
            print("⚠️ Выберите файлы")
            return
        deleted = []
        for path in selected:
            try:
                Path(path).unlink()
                deleted.append(Path(path).stem.rstrip("_"))
                print(f"🗑️ Удалён: {Path(path).stem}")
            except Exception as e:
                print(f"❌ Ошибка {Path(path).stem}: {e}")
        print(f"\n✅ Удалено {len(deleted)} файлов")
        if deleted:
            print(f"📋 Для перегенерации: {', '.join(deleted)}")

select_bad_btn.on_click(on_select_bad)
delete_btn.on_click(on_delete)

tab3_content = widgets.VBox([
    widgets.HTML("<div style='background:#1e1e1e;padding:12px;border-radius:8px;color:#aaa'><small>Зажмите Ctrl/Cmd для выбора нескольких</small></div>"),
    delete_select,
    widgets.HBox([select_bad_btn, delete_btn, regen_btn]),
    delete_out
])

tab_analysis = widgets.Output()
tab_glyphs   = widgets.Output()

with tab_analysis:
    display(HTML(make_analysis_html()))
with tab_glyphs:
    display(HTML(make_all_glyphs_html()))

tabs = widgets.Tab(children=[tab_analysis, tab_glyphs, tab3_content])
tabs.set_title(0, "🔍 Анализ")
tabs.set_title(1, "👁️ Все глифы")
tabs.set_title(2, "🗑️ Удаление")
display(tabs)


---
## Ячейка 5 — Сборка шрифта (OTF + TTF)

In [ ]:
import ufoLib2
import ufo2ft
from pathlib import Path
import re

# ============================================================
# НАСТРОЙКИ ШРИФТА — измените под себя
# ============================================================
FONT_NAME     = "TestFont"
FONT_FAMILY   = "Test Font"
FONT_STYLE    = "Regular"
FONT_DESIGNER = "Your Name"

OUTPUT_FONT_DIR = str(Path(SVG_DIR).parent / "fonts")
Path(OUTPUT_FONT_DIR).mkdir(parents=True, exist_ok=True)

UPM        = 1000
ASCENDER   = 800
DESCENDER  = -200
X_HEIGHT   = 500
CAP_HEIGHT = 700

# ============================================================

def flip_y(y, upm=1000):
    return upm - y

def parse_svg_path(d):
    def get_coords(s):
        return [float(x) for x in re.findall(r"-?\d+\.?\d*", s)]
    tokens = re.findall(r"([MmLlHhVvCcSsQqTtAaZz])([^MmLlHhVvCcSsQqTtAaZz]*)", d)
    segments = []
    current = [0, 0]
    contour_open = False
    for cmd, args in tokens:
        coords = get_coords(args)
        if cmd == "M":
            if contour_open:
                segments.append(("close",))
            pairs = [(coords[i], coords[i+1]) for i in range(0, len(coords)-1, 2)]
            current = list(pairs[0])
            segments.append(("move", pairs[0]))
            contour_open = True
            for p in pairs[1:]:
                segments.append(("line", p))
                current = list(p)
        elif cmd == "L":
            pairs = [(coords[i], coords[i+1]) for i in range(0, len(coords)-1, 2)]
            for p in pairs:
                segments.append(("line", p))
                current = list(p)
        elif cmd == "Q":
            for i in range(0, len(coords)-3, 4):
                cp = (coords[i],   coords[i+1])
                ep = (coords[i+2], coords[i+3])
                segments.append(("qcurve", cp, ep))
                current = list(ep)
        elif cmd == "C":
            for i in range(0, len(coords)-5, 6):
                cp1 = (coords[i],   coords[i+1])
                cp2 = (coords[i+2], coords[i+3])
                ep  = (coords[i+4], coords[i+5])
                segments.append(("curve", cp1, cp2, ep))
                current = list(ep)
        elif cmd in ("Z", "z"):
            segments.append(("close",))
            contour_open = False
    return segments

def build_ufo(svg_map, upm=1000):
    font = ufoLib2.Font()
    font.info.familyName           = FONT_FAMILY
    font.info.styleName            = FONT_STYLE
    font.info.unitsPerEm           = upm
    font.info.ascender             = ASCENDER
    font.info.descender            = DESCENDER
    font.info.xHeight              = X_HEIGHT
    font.info.capHeight            = CAP_HEIGHT
    font.info.copyright            = f"Copyright 2026 {FONT_DESIGNER}"
    font.info.openTypeNameDesigner = FONT_DESIGNER
    font.info.versionMajor         = 1
    font.info.versionMinor         = 0

    notdef = font.newGlyph(".notdef")
    notdef.width = 500
    pen = notdef.getPen()
    pen.moveTo((50, 0))
    pen.lineTo((450, 0))
    pen.lineTo((450, 700))
    pen.lineTo((50, 700))
    pen.closePath()
    pen.moveTo((100, 50))
    pen.lineTo((100, 650))
    pen.lineTo((400, 650))
    pen.lineTo((400, 50))
    pen.closePath()

    unicode_map = {c: ord(c) for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"}
    added = []
    skipped = []
    all_files = [f for files in svg_map.values() for f in files]

    for svg_file in sorted(all_files, key=lambda x: x.stem):
        stem = svg_file.stem
        char = stem.rstrip("_")
        if char not in unicode_map:
            skipped.append(stem)
            continue
        try:
            content = svg_file.read_text()
        except:
            skipped.append(stem)
            continue
        path_matches = re.findall(r'd="([^"]+)"', content)
        if not path_matches:
            skipped.append(stem)
            continue
        glyph_name = f"uni{unicode_map[char]:04X}"
        glyph = font.newGlyph(glyph_name)
        glyph.width = upm
        glyph.unicodes = [unicode_map[char]]
        pen = glyph.getPen()
        contour_open = False
        for path_d in path_matches:
            segments = parse_svg_path(path_d)
            if not segments:
                continue
            for seg in segments:
                if seg[0] == "move":
                    x, y = seg[1]
                    pen.moveTo((x, flip_y(y, upm)))
                    contour_open = True
                elif seg[0] == "line" and contour_open:
                    x, y = seg[1]
                    pen.lineTo((x, flip_y(y, upm)))
                elif seg[0] == "qcurve" and contour_open:
                    cp = (seg[1][0], flip_y(seg[1][1], upm))
                    ep = (seg[2][0], flip_y(seg[2][1], upm))
                    pen.qCurveTo(cp, ep)
                elif seg[0] == "curve" and contour_open:
                    cp1 = (seg[1][0], flip_y(seg[1][1], upm))
                    cp2 = (seg[2][0], flip_y(seg[2][1], upm))
                    ep  = (seg[3][0], flip_y(seg[3][1], upm))
                    pen.curveTo(cp1, cp2, ep)
                elif seg[0] == "close" and contour_open:
                    pen.closePath()
                    contour_open = False
            if contour_open:
                pen.closePath()
                contour_open = False
        added.append(char)
    return font, added, skipped

print("🔨 Собираем шрифт...")
font, added, skipped = build_ufo(svg_map)
print(f"✅ Добавлено глифов : {len(added)} — {', '.join(sorted(added))}")
if skipped:
    print(f"⚠️ Пропущено        : {', '.join(skipped)}")

print(f"\n💾 Компилируем OTF и TTF...")
try:
    ttf_path = f"{OUTPUT_FONT_DIR}/{FONT_NAME}-{FONT_STYLE}.ttf"
    otf_path = f"{OUTPUT_FONT_DIR}/{FONT_NAME}-{FONT_STYLE}.otf"
    ttf = ufo2ft.compileTTF(font, removeOverlaps=False)
    ttf.save(ttf_path)
    print(f"✅ TTF сохранён: {ttf_path}")
    otf = ufo2ft.compileOTF(font, removeOverlaps=False)
    otf.save(otf_path)
    print(f"✅ OTF сохранён: {otf_path}")
    print(f"\n📁 Папка: {OUTPUT_FONT_DIR}")
except Exception as e:
    print(f"❌ Ошибка компиляции: {e}")
    import traceback
    traceback.print_exc()


---
## Ячейка 6 — Визуальный тест шрифта

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

ttf_path = f"{OUTPUT_FONT_DIR}/{FONT_NAME}-{FONT_STYLE}.ttf"
fm.fontManager.addfont(ttf_path)
prop = fm.FontProperties(fname=ttf_path)
print(f"✅ Шрифт загружен: {prop.get_name()}")

fig, axes = plt.subplots(2, 1, figsize=(18, 6))
fig.patch.set_facecolor('#1e1e1e')

test_rows = ["ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz"]
labels    = ["Заглавные", "Строчные"]
colors    = ["#a8d8f0", "#f0d080"]

for ax, text, label, color in zip(axes, test_rows, labels, colors):
    ax.set_facecolor('#1e1e1e')
    ax.text(0.5, 0.5, text, fontproperties=prop, fontsize=36,
            color=color, ha='center', va='center', transform=ax.transAxes)
    ax.set_title(label, color='#888', fontsize=10, pad=4)
    ax.axis('off')

plt.suptitle(f'{FONT_NAME} — визуальный тест', color='#e0e0e0', fontsize=12)
plt.tight_layout()
preview_path = f"{OUTPUT_FONT_DIR}/{FONT_NAME}_preview.png"
plt.savefig(preview_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()
print(f"✅ Превью сохранено: {preview_path}")
